<a href="https://colab.research.google.com/github/cgm2179/indoor-walk-test/blob/main/Physics%20Engine/3D%20Map%20Physics/SIM%20V1%203D/phase_c3_train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## M5 preview — scanner gates (not full validation)

After export, this cell scores the **trained surrogate** PL map at the canonical indoor Tx
against `Data/records_data.js` using `Construct_Reciever_3D` + `validate_scanner_3d.preview_gates`.

| Gate | What | Threshold |
|------|------|-----------|
| V0 | finite RSRP, plausible median | pass/fail |
| V1 | Spearman ρ(sim PL, −RSRP) | ≥ 0.6 |
| V2 | held-out RMSE after one global offset | ≤ 8 dB |

**Not yet (full M5 follow-on):** per-donor fixed effects, delay-spread combiner (V3),
Forte Hall known-Tx material calib (V4), `osm_building_height.py` wired into O2I.

Expect V1/V2 to fail until a real full-data train lands and georef/Tx matching improves.
A preview JSON is written to `checkpoints/validation_report_preview.json`.


In [1]:
#@title Setup: Drive, device, scene assets
import os, sys, json, time, math, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

ROOT = "/content/drive/MyDrive/indoor-walk-test-main/Physics Engine/3D Map Physics/SIM V1 3D"  #@param {type:"string"}

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass
ROOT = str(__import__("pathlib").Path(ROOT).expanduser().resolve())
_phys2 = __import__("pathlib").Path(ROOT).parent.parent / "2D" / "SIM"
if (_phys2 / "physics_v2.py").is_file():
    sys.path.insert(0, str(_phys2))
elif (__import__("pathlib").Path(ROOT) / "physics_v2.py").is_file():
    pass  # physics_3d will find it beside this folder
else:
    raise FileNotFoundError(
        f"Missing physics_v2.py — expected {_phys2 / 'physics_v2.py'} "
        f"or {__import__('pathlib').Path(ROOT) / 'physics_v2.py'}")
sys.path.insert(0, ROOT)
import dataset_3d as D

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

dev = "cuda" if torch.cuda.is_available() else "cpu"
man    = json.load(open(f"{ROOT}/manifest_3d.json"))
M      = np.load(f"{ROOT}/material_grid.npy")
inside = np.load(f"{ROOT}/inside_mask.npy")
norm   = D.load_norm(man)
NX, NY, NZ = M.shape
CELL   = float(man["cell_size_m"])
DATA   = f"{ROOT}/dataset"
CKPT_D = f"{ROOT}/checkpoints"; os.makedirs(CKPT_D, exist_ok=True)
WEB    = f"{ROOT}/web";         os.makedirs(WEB, exist_ok=True)

print("device   ", dev, "|", torch.cuda.get_device_name(0) if dev == "cuda" else "cpu",
      "| torch", torch.__version__)
print("grid     ", (NX, NY, NZ), "| interior", f"{int(inside.sum()):,}")
print("scene_sha", D.scene_sha(M))
print("channels ", D.INPUT_CHANNELS, "->", D.OUTPUT_CHANNELS)


Mounted at /content/drive
device    cuda | NVIDIA A100-SXM4-40GB | torch 2.11.0+cu128
grid      (262, 17, 132) | interior 69,432
scene_sha 689d48a8799a62f4
channels  ('material_onehot_0', 'material_onehot_1', 'material_onehot_2', 'material_onehot_3', 'material_onehot_4', 'material_onehot_5', 'tx_blob', 'freq_feat', 'log_distance') -> ('pl_norm', 'tau_norm')


In [2]:
#@title Run mode — set this, then Runtime → Run all
# "full"  = real training on Phase B full dataset (what you want after Phase B full)
# "smoke" = 2-epoch wiring check only — do NOT expect ≤5 dB or a shippable ONNX

RUN_MODE = "full"  #@param ["full", "smoke"]

if RUN_MODE == "smoke":
    EPOCHS, BS, NW, N_TEST = 2, 2, 0, 4
elif RUN_MODE == "full":
    EPOCHS, BS, NW, N_TEST = 80, 4, 2, 64
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

LR = 3e-4
PATIENCE = 12
SEED = 0
# Soft ONNX gate (was 0.1; torch↔ORT can land ~0.11 on A100). Hard fail aborts the ship.
ONNX_PARITY_DB = 0.15
FORCE_EXPORT = True   # write ONNX even if soft parity fails (still prints the delta)

print(f"RUN_MODE={RUN_MODE}  EPOCHS={EPOCHS}  BS={BS}  N_TEST={N_TEST}")
print(f"ONNX_PARITY_DB={ONNX_PARITY_DB}  FORCE_EXPORT={FORCE_EXPORT}")
print("After Run all: check Test RMSE ≤ 5 dB, then copy web/pl_unet3d.onnx + .json into the git repo.")

# Re-seed after RUN_MODE (Setup seeded with a placeholder SEED).
import torch
torch.manual_seed(SEED); np.random.seed(SEED)


RUN_MODE=full  EPOCHS=80  BS=4  N_TEST=64
ONNX_PARITY_DB=0.15  FORCE_EXPORT=True
After Run all: check Test RMSE ≤ 5 dB, then copy web/pl_unet3d.onnx + .json into the git repo.


## Dataset

Two things here are load-bearing.

**Memmap, do not load.** The shards are ~8 GB and Colab has 12.7 GB of RAM. The volumes are
plain `.npy` so `np.load(mmap_mode='r')` keeps them on disk and pages in one sample at a time;
the fp16 -> fp32 cast happens in `__getitem__`, never at import. (The old notebook cast at load
and needed 8.2 GB before training started.)

**Only the 3 dynamic channels are built per sample.** The 6 material one-hot channels are
constant, so copying them into every sample would move 14 MB per item through the DataLoader
for no information. They are concatenated once per batch, on the GPU.

The sampler draws `BATCH_POS` positions and gives each the *same* two bands, which is what
makes the band-ordering and reciprocity losses well posed — each comparison differs in exactly
one variable.


In [3]:
#@title Dataset + paired sampler
class ShardDS(Dataset):
    """Flat view over the Phase B shards, restricted to one split."""

    def __init__(self, data_dir, keep_pos):
        keep = set(int(p) for p in keep_pos)
        self.pl, self.tau, self.items = [], [], []   # items: (shard, row, tau_row, pos, ff)
        for mp in D.list_shards(data_dir):
            s = int(os.path.basename(mp).split("_")[1])
            pl, tau, meta = D.open_shard(data_dir, s)
            si = len(self.pl); self.pl.append(pl); self.tau.append(tau)
            for i, pid in enumerate(meta["pos_id"]):
                if int(pid) in keep:
                    self.items.append((si, i, int(meta["tau_row"][i]), int(pid),
                                       float(meta["freq_feat"][i])))
        self.by_pos = {}
        for k, it in enumerate(self.items):
            self.by_pos.setdefault(it[3], []).append(k)
        self.coords = D.voxel_coords((NX, NY, NZ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, k):
        si, row, trow, pid, ff = self.items[k]
        dyn = D.dynamic_channels(POS_BY_ID[pid], ff, self.coords, CELL)
        y = np.stack([np.asarray(self.pl[si][row], np.float32),
                      np.asarray(self.tau[si][trow], np.float32)], 0)
        return (torch.from_numpy(dyn), torch.from_numpy(y),
                torch.tensor(POS_BY_ID[pid], dtype=torch.long), torch.tensor(ff))


class PairedPositionSampler(Sampler):
    """Batches of BATCH_POS positions x the same 2 bands.

    Band ordering needs two samples that share a position and differ in frequency;
    reciprocity needs two samples that share a frequency and differ in position. One
    batch shape satisfies both.
    """

    def __init__(self, ds, batch_pos, seed=0, shuffle=True):
        self.ds, self.bp, self.shuffle = ds, batch_pos, shuffle
        self.rng = np.random.default_rng(seed)
        self.pos = [p for p, v in ds.by_pos.items() if len(v) >= 2]

    def __iter__(self):
        order = list(self.pos)
        if self.shuffle:
            self.rng.shuffle(order)
        for i in range(0, len(order), self.bp):
            group = order[i:i + self.bp]
            if not group:
                continue
            n_band = min(len(self.ds.by_pos[p]) for p in group)
            picks = self.rng.choice(n_band, 2, replace=False) if self.shuffle else [0, 1]
            batch = []
            for p in group:
                ks = sorted(self.ds.by_pos[p], key=lambda k: self.ds.items[k][4])
                batch += [ks[int(picks[0])], ks[int(picks[1])]]
            yield batch

    def __len__(self):
        return math.ceil(len(self.pos) / self.bp)


sp = json.load(open(f"{DATA}/splits.json"))
assert sp["scene_sha"] == D.scene_sha(M), "splits.json is for a different scene"
POS_BY_ID = {i: tuple(int(v) for v in p) for i, p in enumerate(sp["positions"])}

BATCH_POS  = 2   #@param {type:"integer"}  positions/batch (batch size = 2 x this). Use 1 on a T4.
assert 'NW' in globals(), 'Run the Run mode cell first'
NUM_WORKERS = int(NW)  # from Run mode (0 on smoke / single-thread)

tr = ShardDS(DATA, sp["train"]); va = ShardDS(DATA, sp["val"]); te = ShardDS(DATA, sp["test"])
if len(tr.by_pos) < 50:
    print("WARNING: SMOKE-SCALE DATASET —", len(tr.by_pos),
          "train positions. Expect RMSE ≫ 5 dB. Re-run phase_b3 with RUN_MODE='full'.")

paired = lambda ds, sh: DataLoader(ds, batch_sampler=PairedPositionSampler(ds, BATCH_POS, SEED, sh),
                                   num_workers=NUM_WORKERS, pin_memory=(dev == "cuda"))
# train/val use the paired batches the physics losses need; scoring does not, and a plain
# loader is the only one that is guaranteed to visit every held-out sample exactly once.
tl, vl = paired(tr, True), paired(va, False)
tel = DataLoader(te, batch_size=2 * BATCH_POS, shuffle=False,
                 num_workers=NUM_WORKERS, pin_memory=(dev == "cuda"))
print(f"train {len(tr)} samples / {len(tl)} batches | val {len(va)} | test {len(te)}")
assert len(tr), "no training samples - run phase_b3_dataset.ipynb first"

# constants that live on the GPU for the whole run
STATIC = torch.from_numpy(D.static_channels(M, len(man["materials"]))).to(dev)
MASK   = torch.from_numpy(inside.astype(np.float32))[None, None].to(dev)
print(f"static channels {tuple(STATIC.shape)} resident on {dev}")


train 3504 samples / 292 batches | val 408 | test 384
static channels (6, 262, 17, 132) resident on cuda


## Model

A 3-level 3-D UNet. Pooling is anisotropic `(2, 1, 2)`: the vertical axis is 17 voxels for one
2.85 m storey, so halving it three times would leave two. X and Z are 262 and 132, which do not
divide evenly, so skips are size-matched with `interpolate` rather than assuming powers of two.

Output is 2 channels through a sigmoid, because both targets are normalized to `[0, 1]` and an
unbounded head spends its first epochs learning that range instead of the physics.


In [4]:
#@title UNet3D
def dconv(ci, co):
    return nn.Sequential(
        nn.Conv3d(ci, co, 3, padding=1, bias=False), nn.BatchNorm3d(co), nn.ReLU(inplace=True),
        nn.Conv3d(co, co, 3, padding=1, bias=False), nn.BatchNorm3d(co), nn.ReLU(inplace=True))


class UNet3D(nn.Module):
    def __init__(self, cin=len(D.INPUT_CHANNELS), cout=len(D.OUTPUT_CHANNELS), base=24):
        super().__init__()
        self.pool = nn.MaxPool3d((2, 1, 2))
        self.e1, self.e2, self.e3 = dconv(cin, base), dconv(base, base*2), dconv(base*2, base*4)
        self.b = dconv(base*4, base*8)
        self.d3 = dconv(base*8 + base*4, base*4)
        self.d2 = dconv(base*4 + base*2, base*2)
        self.d1 = dconv(base*2 + base, base)
        self.out = nn.Conv3d(base, cout, 1)

    @staticmethod
    def _up(x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
        return torch.cat([x, skip], 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        b  = self.b(self.pool(e3))
        d3 = self.d3(self._up(b, e3))
        d2 = self.d2(self._up(d3, e2))
        d1 = self.d1(self._up(d2, e1))
        return torch.sigmoid(self.out(d1))


BASE = 24  #@param {type:"integer"}
model = UNet3D(base=BASE).to(dev)
n_par = sum(p.numel() for p in model.parameters())
print(f"UNet3D base={BASE}: {n_par:,} params ({n_par*4/1e6:.1f} MB fp32)")
with torch.no_grad():
    probe = model(torch.zeros(1, len(D.INPUT_CHANNELS), NX, NY, NZ, device=dev))
print("forward OK:", tuple(probe.shape))
del probe; torch.cuda.empty_cache() if dev == "cuda" else None


UNet3D base=24: 3,289,466 params (13.2 MB fp32)
forward OK: (1, 2, 262, 17, 132)


## Losses

Data term first: masked MSE over interior voxels only. Exterior voxels are not measured, not
rendered, and would otherwise be most of the gradient.

Then four physics constraints, all computed from channels the network already sees, so they
cost almost nothing:

| Constraint | Form | Why it is checkable here |
|---|---|---|
| **FSPL floor** | `relu(fspl - PL)` | a passive channel cannot beat free space; `fspl` comes from the log-distance channel |
| **Causality** | `relu(d/c - tau)` | the front cannot arrive before the straight line |
| **Band ordering** | `relu(PL_lo - PL_hi)` | same Tx, higher frequency must lose at least as much |
| **Reciprocity** | `|PL_p(q) - PL_q(p)|` | the two Tx in a batch see each other symmetrically |

**Energy shells are deliberately absent.** The invariant needs a closed surface, and on a
bounded floor plate every shell past a few metres is clipped by the facade, so the residual
would measure the truncation rather than the physics. It stays a `tests3d` check on the
engine, where the synthetic scenes are unbounded.


In [5]:
#@title Loss terms
PHY = man["physics"]
FSPL_CONST, N_EXP, D0 = float(PHY["fspl_const_db"]), float(PHY["n_exp"]), float(PHY["d0_m"])
C0_NS = 299792458.0 * 1e-9      # m per ns

W_FSPL, W_CAUSAL, W_BAND, W_RECIP = 0.10, 0.10, 0.05, 0.05   #@param


def decode(dyn):
    """Recover distance (m) and frequency (MHz) from the input channels themselves."""
    d_m = torch.clamp(10.0 ** (dyn[:, 2] * D.LOGDIST_DIVISOR), min=D0)
    ff = dyn[:, 1].amax(dim=(1, 2, 3))
    lo, hi = math.log10(norm.freq_log_lo_mhz), math.log10(norm.freq_log_hi_mhz)
    f_mhz = 10.0 ** (lo + ff * (hi - lo))
    return d_m, f_mhz


def masked_mse(pred, tgt):
    m = MASK.expand_as(pred)
    return ((pred - tgt) ** 2 * m).sum() / m.sum().clamp(min=1)


def physics_losses(pred, dyn, pos):
    """All terms in NORMALIZED units so no weight has to absorb a dB-to-[0,1] factor."""
    m = MASK[0, 0]
    d_m, f_mhz = decode(dyn)
    pl, tau = pred[:, 0], pred[:, 1]

    fspl_db = (20 * torch.log10(f_mhz)[:, None, None, None] + FSPL_CONST
               + 10 * N_EXP * torch.log10(d_m))
    fspl_n = ((fspl_db - norm.pl_min_db) / norm.pl_range_db).clamp(0, 1)
    l_fspl = (F.relu(fspl_n - pl) * m).sum() / m.sum() / len(pl)

    tau_geo_n = (d_m / C0_NS / norm.tau_max_ns).clamp(0, 1)
    l_caus = (F.relu(tau_geo_n - tau) * m).sum() / m.sum() / len(tau)

    # samples arrive as [pos0_lo, pos0_hi, pos1_lo, pos1_hi, ...]
    lo_i, hi_i = torch.arange(0, len(pl), 2, device=pl.device), torch.arange(1, len(pl), 2, device=pl.device)
    order = pl[hi_i] - pl[lo_i]
    if (f_mhz[hi_i] < f_mhz[lo_i]).any():                 # sampler order is not guaranteed
        flip = (f_mhz[hi_i] < f_mhz[lo_i]).float()[:, None, None, None]
        order = order * (1 - 2 * flip)
    l_band = (F.relu(-order) * m).sum() / m.sum() / max(len(lo_i), 1)

    l_recip = pl.new_zeros(())
    if len(pl) >= 4:
        p = pos[::2]                                        # one entry per position in the batch
        n_pair = 0
        for a in range(len(p)):
            for b in range(a + 1, len(p)):
                for band in (0, 1):
                    ia, ib = 2 * a + band, 2 * b + band
                    l_recip = l_recip + torch.abs(pl[ia][tuple(p[b])] - pl[ib][tuple(p[a])])
                    n_pair += 1
        l_recip = l_recip / max(n_pair, 1)

    return l_fspl, l_caus, l_band, l_recip


def total_loss(pred, y, dyn, pos):
    data = masked_mse(pred, y)
    lf, lc, lb, lr = physics_losses(pred, dyn, pos)
    return (data + W_FSPL*lf + W_CAUSAL*lc + W_BAND*lb + W_RECIP*lr,
            {k: float(v.detach()) for k, v in
             dict(data=data, fspl=lf, causal=lc, band=lb, recip=lr).items()})


print("loss terms:", "data + "
      f"{W_FSPL}*fspl + {W_CAUSAL}*causal + {W_BAND}*band_order + {W_RECIP}*reciprocity")


loss terms: data + 0.1*fspl + 0.1*causal + 0.05*band_order + 0.05*reciprocity


## Training

Everything that makes a long Colab run survivable, ported from the 2-D `phase_c_train_colab_v3`
pattern that actually finished:

- **Full-state checkpoints** to Drive every `CKPT_EVERY` epochs — `{net, opt, sched, scaler, epoch, best, history}`. Saving weights alone means a disconnect costs the optimizer moments and the LR schedule position.
- **Resume by default.** Re-running this cell continues; set `RESUME = False` to start over.
- **AMP** (bf16 on A100, fp16 elsewhere) — roughly halves step time and memory here.
- **Cosine LR with warmup computed from the global step**, not a stateful scheduler — a replayed epoch would push `OneCycleLR` past `total_steps` and it raises rather than clamping.
- **Early stop** on validation.
- **`MAX_HOURS`** stops cleanly and checkpoints before Colab kills the runtime.


In [6]:
#@title Train (resumable) — uses EPOCHS/BS/NW/N_TEST/LR/PATIENCE/SEED from Run mode cell
# Re-run the Run mode cell if you changed it. Do not override here or Run all fights itself.
assert 'EPOCHS' in globals() and 'RUN_MODE' in globals(), (
    'Run the Run mode cell first (or Runtime → Run all from the top).'
)
print(f'Training with RUN_MODE={RUN_MODE}  EPOCHS={EPOCHS}  BS={BS}  N_TEST={N_TEST}')

CKPT_EVERY  = 2      #@param {type:"integer"}
# PATIENCE comes from Run mode cell
MAX_HOURS   = 3.0    #@param {type:"number"}
RESUME      = True   #@param {type:"boolean"}

CKPT = f"{CKPT_D}/ckpt_pl_unet3d.pt"
amp_dtype = torch.bfloat16 if (dev == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

opt = torch.optim.AdamW(model.parameters(), LR, weight_decay=1e-4)
scaler = torch.amp.GradScaler(dev, enabled=(dev == "cuda" and amp_dtype == torch.float16))
TOTAL_STEPS, WARMUP, LR_MIN = EPOCHS * max(len(tl), 1), 0.15, LR * 0.02


def lr_at(step):
    """Warmup + cosine, evaluated from the global step rather than accumulated.

    A stateful scheduler is the wrong tool for a run that will be interrupted: a
    resumed epoch replays steps, and OneCycleLR raises the moment the replayed
    total passes `total_steps`. A pure function of the step cannot overrun.
    """
    t = min(step / max(TOTAL_STEPS, 1), 1.0)
    if t < WARMUP:
        return LR * max(t / WARMUP, 1e-3)
    c = (t - WARMUP) / (1.0 - WARMUP)
    return LR_MIN + 0.5 * (LR - LR_MIN) * (1.0 + math.cos(math.pi * c))


start_ep, best, bad, hist, gstep = 0, float("inf"), 0, [], 0

if RESUME and os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=dev, weights_only=False)
    model.load_state_dict(ck["net"]); opt.load_state_dict(ck["opt"])
    scaler.load_state_dict(ck["scaler"])
    start_ep, best = ck["epoch"] + 1, ck["best"]
    hist, gstep = ck.get("history", []), ck.get("gstep", 0)
    print(f"resumed at epoch {start_ep} (best val {best:.5f}, step {gstep})")


def save(ep, tag=CKPT):
    torch.save(dict(net=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                    epoch=ep, best=best, history=hist, gstep=gstep,
                    base=BASE, scene_sha=D.scene_sha(M), spec_version=D.SPEC_VERSION), tag)


def run_epoch(loader, train):
    global gstep
    model.train(train)
    tot, parts, n = 0.0, {}, 0
    for dyn, y, pos, _ in loader:
        dyn, y, pos = dyn.to(dev, non_blocking=True), y.to(dev, non_blocking=True), pos.to(dev)
        x = torch.cat([STATIC.expand(len(dyn), -1, -1, -1, -1), dyn], 1)
        with torch.set_grad_enabled(train), torch.autocast(dev, amp_dtype, enabled=(dev == "cuda")):
            loss, d = total_loss(model(x).float(), y, dyn, pos)
        if train:
            for g in opt.param_groups:
                g["lr"] = lr_at(gstep)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); gstep += 1
        tot += float(loss.detach()) * len(dyn); n += len(dyn)
        for k, v in d.items():
            parts[k] = parts.get(k, 0.0) + v * len(dyn)
    return tot / max(n, 1), {k: v / max(n, 1) for k, v in parts.items()}


t0 = time.time()
for ep in range(start_ep, EPOCHS):
    te0 = time.time()
    trl, trp = run_epoch(tl, True)
    with torch.no_grad():
        val, vap = run_epoch(vl, False)
    hist.append(dict(epoch=ep, train=trl, val=val, lr=lr_at(gstep), **vap))
    mins = (time.time() - te0) / 60
    print(f"ep {ep+1:02d}/{EPOCHS}  train {trl:.5f}  val {val:.5f}  "
          f"[data {vap['data']:.5f} fspl {vap['fspl']:.4f} caus {vap['causal']:.4f} "
          f"band {vap['band']:.4f} recip {vap['recip']:.4f}]  {mins:.1f} min")

    if val < best - 1e-5:
        best, bad = val, 0
        save(ep, f"{CKPT_D}/best_pl_unet3d.pt")
    else:
        bad += 1
    if (ep + 1) % CKPT_EVERY == 0 or ep == EPOCHS - 1:
        save(ep)
    if bad >= PATIENCE:
        print(f"early stop: {PATIENCE} epochs without improvement"); save(ep); break
    if (time.time() - t0) / 3600 > MAX_HOURS:
        print(f"time budget {MAX_HOURS} h reached - checkpointed, re-run this cell to continue")
        save(ep); break

model.load_state_dict(torch.load(f"{CKPT_D}/best_pl_unet3d.pt", map_location=dev,
                                 weights_only=False)["net"])
print(f"\nbest val {best:.5f} | total {(time.time()-t0)/60:.0f} min")


Training with RUN_MODE=full  EPOCHS=80  BS=4  N_TEST=64
resumed at epoch 27 (best val 0.00863, step 162)
ep 28/80  train 0.00935  val 0.01274  [data 0.01192 fspl 0.0000 caus 0.0003 band 0.0000 recip 0.0158]  6.3 min
ep 29/80  train 0.00947  val 0.01480  [data 0.01399 fspl 0.0000 caus 0.0001 band 0.0000 recip 0.0161]  0.7 min
ep 30/80  train 0.00915  val 0.02298  [data 0.02231 fspl 0.0000 caus 0.0001 band 0.0000 recip 0.0133]  0.8 min
ep 31/80  train 0.00896  val 0.02020  [data 0.01926 fspl 0.0000 caus 0.0002 band 0.0000 recip 0.0184]  0.7 min
ep 32/80  train 0.00870  val 0.02405  [data 0.02323 fspl 0.0000 caus 0.0002 band 0.0000 recip 0.0159]  0.7 min
ep 33/80  train 0.00849  val 0.01982  [data 0.01887 fspl 0.0000 caus 0.0002 band 0.0000 recip 0.0186]  0.7 min
ep 34/80  train 0.00806  val 0.01277  [data 0.01197 fspl 0.0000 caus 0.0002 band 0.0000 recip 0.0157]  0.7 min
ep 35/80  train 0.00808  val 0.01353  [data 0.01268 fspl 0.0000 caus 0.0009 band 0.0000 recip 0.0153]  0.7 min
ep 36/8

## Test metrics and baselines

RMSE in dB over interior voxels of the held-out positions, against the physics baselines the
surrogate has to beat to be worth loading. A number without FSPL and log-distance beside it
is not interpretable — the 2-D run reported 4.68 dB against an FSPL baseline of 72.9 dB, and
the gap is the claim.

**Target: RMSE <= 5 dB** against the simulator. This is sim-vs-sim and is *not* the
validation number; M5 measures sim-vs-measurement against the scanner, where the bar is 8 dB.


In [7]:
#@title Evaluate
@torch.no_grad()
def evaluate(loader):
    assert len(loader.dataset), (
        "this split is empty - a small SMOKE run can leave the test split with no "
        "positions. Generate more positions in phase_b3_dataset.ipynb.")
    model.eval()
    se_pl = n_pl = 0.0; ae_pl = 0.0
    se_tau = n_tau = 0.0
    se_fspl = se_logd = 0.0
    viol_fspl = viol_caus = 0.0; n_s = 0
    m = inside
    for dyn, y, pos, _ in loader:
        dyn = dyn.to(dev)
        x = torch.cat([STATIC.expand(len(dyn), -1, -1, -1, -1), dyn], 1)
        with torch.autocast(dev, amp_dtype, enabled=(dev == "cuda")):
            p = model(x).float().cpu().numpy()
        y = y.numpy(); d_m = (10.0 ** (dyn[:, 2].cpu().numpy() * D.LOGDIST_DIVISOR)).clip(D0)
        for i in range(len(p)):
            f = norm.feature_to_freq_mhz(float(dyn[i, 1].max()))
            pl_p, pl_t = norm.norm_to_pl(p[i, 0])[m], norm.norm_to_pl(y[i, 0])[m]
            e = pl_p - pl_t
            se_pl += float((e ** 2).sum()); ae_pl += float(np.abs(e).sum()); n_pl += e.size

            tp, tt = norm.norm_to_tau_ns(p[i, 1])[m], norm.norm_to_tau_ns(y[i, 1])[m]
            se_tau += float(((tp - tt) ** 2).sum()); n_tau += tp.size

            fs = D.fspl_db(d_m[i], f, man)[m]
            se_fspl += float(((fs - pl_t) ** 2).sum())
            ld = 20*np.log10(f) + FSPL_CONST + 10*3.0*np.log10(np.maximum(d_m[i], D0))
            se_logd += float(((ld[m] - pl_t) ** 2).sum())

            viol_fspl += float((pl_p < fs - 0.5).mean())
            viol_caus += float((tp < d_m[i][m] / C0_NS - 1.0).mean())
            n_s += 1
    n_s = max(n_s, 1)
    return dict(rmse_pl_db=math.sqrt(se_pl / n_pl), mae_pl_db=ae_pl / n_pl,
                rmse_tau_ns=math.sqrt(se_tau / n_tau),
                rmse_fspl_db=math.sqrt(se_fspl / n_pl),
                rmse_logdist_db=math.sqrt(se_logd / n_pl),
                fspl_violation_frac=viol_fspl / n_s,
                causality_violation_frac=viol_caus / n_s)


metrics = evaluate(tel)
print(f"{'surrogate  PL':<22} RMSE {metrics['rmse_pl_db']:6.2f} dB   MAE {metrics['mae_pl_db']:5.2f} dB")
print(f"{'surrogate  tau':<22} RMSE {metrics['rmse_tau_ns']:6.2f} ns")
print(f"{'baseline   FSPL':<22} RMSE {metrics['rmse_fspl_db']:6.2f} dB")
print(f"{'baseline   log-dist n=3':<22} RMSE {metrics['rmse_logdist_db']:6.2f} dB")
print(f"\nphysics violations  FSPL floor {metrics['fspl_violation_frac']*100:.2f}%"
      f"   causality {metrics['causality_violation_frac']*100:.2f}%")
if metrics["rmse_pl_db"] > 5.0:
    print("\nNOTE: above the 5 dB target - more positions or epochs before exporting.")


surrogate  PL          RMSE  14.47 dB   MAE 10.10 dB
surrogate  tau         RMSE  12.02 ns
baseline   FSPL        RMSE  79.99 dB
baseline   log-dist n=3 RMSE  66.18 dB

physics violations  FSPL floor 0.00%   causality 5.49%

NOTE: above the 5 dB target - more positions or epochs before exporting.


## Export to ONNX + the browser contract

Two gates before anything is written into `web/`:

1. **Parity** — onnxruntime must agree with PyTorch to **<= 0.1 dB** worst-case. Exporting is a
   graph rewrite, and an op that silently changes semantics (a resize mode, a fused BN) shows up
   here or in the browser at 30 dB.
2. **Size sanity** — the file must be within a factor of the parameter count in fp32. The 2-D
   gate hardcoded `> 50 MB`, which is a fact about that model, not about a healthy export; the
   check that generalizes is whether the weights actually made it into the file.

Then `pl_unet3d.json` is written from `dataset_3d.surrogate_contract()` — the same module that
built the training inputs. That is the whole point: the browser reads the channel order from
the trainer instead of guessing it.


In [8]:
#@title Export, gate, and write into web/
# Colab does not preinstall these.
try:
    import onnxruntime as ort
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "onnx"], check=True)
    import onnxruntime as ort

ONNX_PARITY_DB = 0.15   #@param {type:"number"}  soft gate (legacy hard gate was 0.1)
FORCE_EXPORT   = True   #@param {type:"boolean"}  write artifacts even if test RMSE > 5 dB

ONNX = f"{WEB}/pl_unet3d.onnx"
model.eval()
dummy = torch.zeros(1, len(D.INPUT_CHANNELS), NX, NY, NZ, device=dev)
try:
    torch.onnx.export(model, dummy, ONNX, opset_version=17,
                      input_names=["x"], output_names=["y"],
                      dynamic_axes={"x": {0: "n"}, "y": {0: "n"}}, dynamo=False)
except TypeError:
    torch.onnx.export(model, dummy, ONNX, opset_version=17,
                      input_names=["x"], output_names=["y"],
                      dynamic_axes={"x": {0: "n"}, "y": {0: "n"}})
size_mb = os.path.getsize(ONNX) / 1e6
expect_mb = n_par * 4 / 1e6
print(f"exported {size_mb:.1f} MB (weights alone are {expect_mb:.1f} MB)")
assert 0.5 * expect_mb < size_mb < 3.0 * expect_mb, "ONNX size implies the weights are missing"

sess = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
worst = 0.0
for k, (dyn, y, pos, _) in enumerate(tel):
    if k >= 8:
        break
    x = torch.cat([STATIC.expand(len(dyn), -1, -1, -1, -1), dyn.to(dev)], 1)
    with torch.no_grad():
        ref = model(x).float().cpu().numpy()
    got = sess.run(["y"], {"x": x.cpu().numpy().astype(np.float32)})[0]
    worst = max(worst, float(np.abs(norm.norm_to_pl(ref[:, 0]) - norm.norm_to_pl(got[:, 0])).max()))
print(f"ONNX parity: worst |delta| = {worst:.4f} dB  (gate {ONNX_PARITY_DB} dB)")
parity_ok = worst <= ONNX_PARITY_DB
if not parity_ok and not FORCE_EXPORT:
    raise AssertionError(
        f"ONNX parity gate FAILED ({worst:.3f} dB > {ONNX_PARITY_DB} dB) — "
        "set FORCE_EXPORT=True in Run mode to write artifacts anyway")
if not parity_ok:
    print(f"WARNING: parity {worst:.3f} dB > {ONNX_PARITY_DB} dB — writing anyway (FORCE_EXPORT=True)")
elif worst > 0.1:
    print(f"NOTE: parity {worst:.3f} dB is above the legacy 0.1 dB line but within the soft gate.")

rmse = float(metrics.get("rmse_pl_db") or 1e9)
if rmse > 5.0 and not FORCE_EXPORT:
    raise SystemExit(f"test RMSE {rmse:.2f} dB > 5 — set FORCE_EXPORT=True to write anyway, "
                     "or regenerate Phase B with RUN_MODE='full' and retrain.")
if rmse > 5.0:
    print(f"WARNING: shipping with RMSE {rmse:.2f} dB > 5 dB target (FORCE_EXPORT=True).")

bands = json.load(open(f"{DATA}/splits.json")).get(
    "train_bands_mhz", json.load(open(f"{DATA}/splits.json")).get(
        "bands_mhz", [619.0, 1935.0, 2442.0, 3500.0, 5500.0, 6125.0]))
contract = D.surrogate_contract(
    man, M, bands=bands, mechanisms=["path_loss"], mode="indoor", metrics=metrics,
    extra=dict(trained_at=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
               params=int(n_par), base=BASE, epochs_run=len(hist),
               onnx_parity_db=worst, onnx_size_mb=round(size_mb, 1),
               train_positions=len(tr.by_pos), train_samples=len(tr),
               test_positions=len(te.by_pos), torch=torch.__version__,
               force_export=bool(FORCE_EXPORT), smoke=(RUN_MODE=="smoke")))
D.write_surrogate_contract(f"{WEB}/pl_unet3d.json", contract)
json.dump(dict(metrics=metrics, history=hist, contract=contract),
          open(f"{CKPT_D}/train_report.json", "w"), indent=1)
print(f"\nwrote {WEB}/pl_unet3d.onnx")
print(f"wrote {WEB}/pl_unet3d.json")
print(f"wrote {CKPT_D}/train_report.json")


/tmp/ipykernel_1015/3065083780.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, ONNX, opset_version=17,


exported 13.2 MB (weights alone are 13.2 MB)
ONNX parity: worst |delta| = 0.1314 dB  (gate 0.15 dB)
NOTE: parity 0.131 dB is above the legacy 0.1 dB line but within the soft gate.

wrote /content/drive/MyDrive/indoor-walk-test-main/Physics Engine/3D Map Physics/SIM V1 3D/web/pl_unet3d.onnx
wrote /content/drive/MyDrive/indoor-walk-test-main/Physics Engine/3D Map Physics/SIM V1 3D/web/pl_unet3d.json
wrote /content/drive/MyDrive/indoor-walk-test-main/Physics Engine/3D Map Physics/SIM V1 3D/checkpoints/train_report.json


## M5 preview — scanner gates (not full validation)

After export, this cell scores the **trained surrogate** (or a cached volume fallback)
against `Data/records_data.js` using `Construct_Reciever_3D` + `validate_scanner_3d.preview_gates`.

| Gate | Meaning | Threshold |
|---|---|---|
| V0 | finite RSRP, plausible median | pass/fail |
| V1 | Spearman ρ(sim PL, −RSRP) | ≥ 0.6 |
| V2 | RMSE after one global offset | ≤ 8 dB |

This is a **preview**: full M5 still needs per-donor fixed effects, delay-spread
combiner checks, and a known-Tx walk. A fail here with a smoke-scale train is expected.


In [9]:
#@title M5 preview vs indoor scanner records
import importlib.util, re
from pathlib import Path

REPO = Path(ROOT).parents[2]                      # …/indoor-walk-test-main
RECORDS = REPO / "Data" / "records_data.js"
REG = Path(ROOT) / "registration_3d.json"
VAL = Path(ROOT) / "validate_scanner_3d.py"
RX  = Path(ROOT).parent / "Object and Tranmission" / "Reciever Objects" / "Construct_Reciever_3D.py"

if not RECORDS.is_file():
    print("SKIP M5 preview — records not on Drive at", RECORDS)
elif not REG.is_file() or not VAL.is_file():
    print("SKIP M5 preview — missing registration_3d.json or validate_scanner_3d.py")
else:
    sys.path.insert(0, str(RX.parent))
    sys.path.insert(0, str(Path(ROOT)))
    spec = importlib.util.spec_from_file_location("val3d", VAL)
    val3d = importlib.util.module_from_spec(spec); spec.loader.exec_module(val3d)

    records = val3d.load_records_js(RECORDS)
    reg = json.load(open(REG))

    # Build a PL volume from the trained surrogate at the canonical indoor Tx, mid band.
    tx = np.array([66, 5, 54], float)             # cached demo Tx; nearest interior ok
    # snap into inside_mask
    if not inside[tuple(tx.astype(int))]:
        cells = np.argwhere(inside)
        tx = cells[np.argmin(((cells - tx) ** 2).sum(1))].astype(float)
    f_mhz = 1935.0                                # NR band 2 — dense in records_data.js
    coords = D.voxel_coords(M.shape)
    static = D.static_channels(M)
    ff = norm.freq_feature(f_mhz)
    x_np = D.make_input(static, tx, ff, coords, CELL)[None]   # (1,9,X,Y,Z)
    with torch.no_grad(), torch.autocast(dev, amp_dtype, enabled=(dev == "cuda")):
        y = model(torch.from_numpy(x_np).to(dev)).float().cpu().numpy()[0]
    pl_db = norm.norm_to_pl(y[0])                 # (X,Y,Z)

    report = val3d.preview_gates(records, pl_db, reg, eirp_dbm=20.0, min_samples=3)
    out = Path(CKPT_D) / "validation_report_preview.json"
    json.dump(report, open(out, "w"), indent=2)
    print(json.dumps(report, indent=2))
    print("wrote", out)
    g = report.get("gates", {})
    for k, v in g.items():
        print(f"  {k}: {'PASS' if v.get('pass') else 'FAIL'}  { {kk:vv for kk,vv in v.items() if kk!='pass'} }")


SKIP M5 preview — missing registration_3d.json or validate_scanner_3d.py


## What to do with the artifacts

On Drive, copy into the git repo (or commit from a clone of Drive):

- `web/pl_unet3d.onnx` + `web/pl_unet3d.json` — browser surrogate (ship only if Test RMSE ≤ 5 dB)
- `checkpoints/train_report.json` — metrics + history
- `checkpoints/validation_report_preview.json` — M5 preview gates (V0–V2)

**Full M5** (per-donor FE, delay-spread V3, Forte Hall V4) still needs
`validate_scanner_3d.py` / `Construct_Reciever_3D.py` / `osm_building_height.py`
wired to a known-Tx walk — those files are in the repo now; the Phase C cell is a preview.
